In [1]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")

# ── Load & prepare data ──────────────────────────────────────────────
df = pd.read_excel('airline_ticket_dataset.xlsx')

lcc_airlines = ['WN', 'NK', 'F9', 'G4', 'SY', 'B6']

df['lcc_present'] = (
    df['carrier_lg'].isin(lcc_airlines) | df['carrier_low'].isin(lcc_airlines)
)
df['market_structure'] = pd.cut(
    df['large_ms'],
    bins=[0, 0.5, 0.75, 1.0],
    labels=['Competitive (<50%)', 'Moderate (50-75%)', 'Dominant (>75%)']
)
df['fare_per_mile'] = df['fare'] / df['nsmiles']
df['dominant_type'] = np.where(
    df['carrier_lg'].isin(lcc_airlines), 'LCC', 'Legacy'
)
df['route'] = df['city1'] + '  ↔  ' + df['city2']

# ── Train predictive model (for Fare Predictor panel) ────────────────
features = ['nsmiles', 'large_ms', 'lf_ms', 'passengers', 'quarter']
ml_data = df.dropna(subset=features + ['fare'])
X = ml_data[features]
y = ml_data['fare']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# ── Precompute dropdown options ──────────────────────────────────────
all_cities = sorted(set(df['city1'].unique()) | set(df['city2'].unique()))
all_carriers = sorted(
    set(df['carrier_lg'].dropna().unique()) | set(df['carrier_low'].dropna().unique())
)

print(f"Loaded {len(df):,} routes across {len(all_cities)} cities")
print(f"Random Forest model trained (R-squared = {rf_model.score(X_test, y_test):.3f})")
print("Dashboard ready -- run the cells below")

Loaded 14,004 routes across 136 cities
Random Forest model trained (R-squared = 0.787)
Dashboard ready -- run the cells below


In [6]:
# ═══════════════════════════════════════════════════════════════════════
# PANEL 1 — Route Explorer
# Select an origin city to browse all routes, fares, and market info.
# ═══════════════════════════════════════════════════════════════════════

city_dropdown = widgets.Dropdown(
    options=all_cities,
    description='Origin city:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='60%')
)

sort_dropdown = widgets.Dropdown(
    options=[
        ('Fare (low → high)', 'fare_asc'),
        ('Fare (high → low)', 'fare_desc'),
        ('Distance (short → long)', 'dist_asc'),
        ('Distance (long → short)', 'dist_desc'),
        ('Passengers (most first)', 'pax_desc'),
    ],
    value='fare_asc',
    description='Sort by:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='40%')
)

route_output = widgets.Output()

def update_route_explorer(*_):
    city = city_dropdown.value
    sort_key = sort_dropdown.value

    subset = df[(df['city1'] == city) | (df['city2'] == city)].copy()

    sort_map = {
        'fare_asc': ('fare', True),
        'fare_desc': ('fare', False),
        'dist_asc': ('nsmiles', True),
        'dist_desc': ('nsmiles', False),
        'pax_desc': ('passengers', False),
    }
    col, asc = sort_map[sort_key]
    subset = subset.sort_values(col, ascending=asc)

    with route_output:
        clear_output(wait=True)
        if subset.empty:
            print(f"No routes found for {city}")
            return

        print(f"Routes from/to: {city}  ({len(subset)} records)\n")
        display(
            subset[['city1', 'city2', 'nsmiles', 'passengers', 'fare',
                     'fare_per_mile', 'carrier_lg', 'large_ms',
                     'carrier_low', 'lf_ms', 'market_structure',
                     'lcc_present', 'quarter']]
            .reset_index(drop=True)
            .style
            .format({
                'fare': '${:.2f}',
                'fare_per_mile': '${:.4f}',
                'large_ms': '{:.1%}',
                'lf_ms': '{:.1%}',
            })
            .set_caption(f"Route details for {city}")
        )

        fig, axes = plt.subplots(1, 2, figsize=(14, 4))

        top = subset.drop_duplicates('route').nlargest(10, 'fare')
        sns.barplot(data=top, y='route', x='fare', hue='route',
                    palette='YlOrRd_r', ax=axes[0], legend=False)
        axes[0].set_title('Top 10 Most Expensive Routes', fontweight='bold')
        axes[0].set_xlabel('Fare ($)')
        axes[0].set_ylabel('')

        all_structures = ['Competitive (<50%)', 'Moderate (50-75%)', 'Dominant (>75%)']
        structure_counts = subset['market_structure'].value_counts()
        structure_counts = structure_counts.reindex(all_structures, fill_value=0)
        colors = sns.color_palette('Set2', len(all_structures))
        bars = axes[1].barh(all_structures, structure_counts.values, color=colors)
        axes[1].set_title('Market Structure Breakdown', fontweight='bold')
        axes[1].set_xlabel('Number of Routes')
        total = structure_counts.sum()
        for bar, count in zip(bars, structure_counts.values):
            pct = count / total * 100 if total > 0 else 0
            label = f'{count}  ({pct:.0f}%)'
            axes[1].text(bar.get_width() + max(total * 0.02, 0.3),
                         bar.get_y() + bar.get_height() / 2,
                         label, va='center', fontsize=10)

        plt.tight_layout()
        plt.show()

city_dropdown.observe(update_route_explorer, names='value')
sort_dropdown.observe(update_route_explorer, names='value')

display(widgets.HBox([city_dropdown, sort_dropdown]))
display(route_output)
update_route_explorer()

Output()

In [3]:
# ═══════════════════════════════════════════════════════════════════════
# PANEL 2 — LCC Penetration vs Fare
# Interactive scatter showing how LCC market share relates to fares.
# Filter by distance range and quarter.
# ═══════════════════════════════════════════════════════════════════════

dist_slider = widgets.IntRangeSlider(
    value=[int(df['nsmiles'].min()), int(df['nsmiles'].max())],
    min=int(df['nsmiles'].min()),
    max=int(df['nsmiles'].max()),
    step=50,
    description='Distance (mi):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='60%')
)

quarter_selector = widgets.SelectMultiple(
    options=[1, 2, 3, 4],
    value=[1, 2, 3, 4],
    description='Quarter(s):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='20%', height='90px')
)

metric_toggle = widgets.ToggleButtons(
    options=['fare', 'fare_per_mile'],
    value='fare',
    description='Y-axis:',
    style={'description_width': 'initial'},
    button_style='info'
)

lcc_output = widgets.Output()

def update_lcc_scatter(*_):
    lo, hi = dist_slider.value
    quarters = list(quarter_selector.value)
    y_col = metric_toggle.value

    subset = df[
        (df['nsmiles'] >= lo) &
        (df['nsmiles'] <= hi) &
        (df['quarter'].isin(quarters))
    ].dropna(subset=['lf_ms', y_col])

    with lcc_output:
        clear_output(wait=True)

        if subset.empty:
            print("No data matches the current filters.")
            return

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        axes[0].scatter(subset['lf_ms'], subset[y_col],
                        alpha=0.25, s=12, c='steelblue')
        z = np.polyfit(subset['lf_ms'], subset[y_col], 1)
        p = np.poly1d(z)
        x_line = np.linspace(subset['lf_ms'].min(), subset['lf_ms'].max(), 100)
        axes[0].plot(x_line, p(x_line), color='red', linewidth=2,
                     linestyle='--', label='Trend')
        y_label = 'Fare ($)' if y_col == 'fare' else 'Fare per Mile ($)'
        axes[0].set_xlabel('LCC Market Share', fontsize=12)
        axes[0].set_ylabel(y_label, fontsize=12)
        axes[0].set_title('LCC Penetration vs Fare', fontsize=14, fontweight='bold')
        axes[0].legend()

        corr = subset['lf_ms'].corr(subset[y_col])
        axes[0].text(0.02, 0.95, f'r = {corr:.3f}',
                     transform=axes[0].transAxes, fontsize=11,
                     verticalalignment='top',
                     bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

        bins = pd.cut(subset['lf_ms'], bins=[0, 0.1, 0.2, 0.3, 0.5, 1.0],
                       labels=['0-10%', '10-20%', '20-30%', '30-50%', '50-100%'])
        bin_means = subset.groupby(bins, observed=True)[y_col].mean()
        bin_means.plot(kind='bar', ax=axes[1], color=sns.color_palette('viridis', len(bin_means)))
        axes[1].set_title(f'Avg {y_label} by LCC Share Bucket', fontsize=14, fontweight='bold')
        axes[1].set_xlabel('LCC Market Share Bucket', fontsize=12)
        axes[1].set_ylabel(y_label, fontsize=12)
        axes[1].tick_params(axis='x', rotation=0)

        for i, v in enumerate(bin_means):
            fmt = f'${v:.2f}' if y_col == 'fare' else f'${v:.4f}'
            axes[1].text(i, v + v * 0.01, fmt, ha='center', fontsize=9)

        plt.tight_layout()
        plt.show()

        print(f"\nShowing {len(subset):,} routes  |  "
              f"Distance: {lo}–{hi} mi  |  "
              f"Quarter(s): {quarters}")

dist_slider.observe(update_lcc_scatter, names='value')
quarter_selector.observe(update_lcc_scatter, names='value')
metric_toggle.observe(update_lcc_scatter, names='value')

display(widgets.VBox([
    metric_toggle,
    widgets.HBox([dist_slider, quarter_selector]),
]))
display(lcc_output)
update_lcc_scatter()

Output()

In [4]:
# ═══════════════════════════════════════════════════════════════════════
# PANEL 3 — Market Structure Comparison
# Compare fares across market structures with interactive filters.
# ═══════════════════════════════════════════════════════════════════════

ms_metric = widgets.ToggleButtons(
    options=[('Average Fare', 'fare'), ('Fare per Mile', 'fare_per_mile')],
    value='fare',
    description='Metric:',
    style={'description_width': 'initial'},
    button_style='info'
)

ms_lcc_filter = widgets.ToggleButtons(
    options=[('All Routes', 'all'), ('LCC Present', 'lcc'), ('Legacy Only', 'legacy')],
    value='all',
    description='Filter:',
    style={'description_width': 'initial'},
    button_style='warning'
)

ms_quarter = widgets.Dropdown(
    options=[('All Quarters', 0), ('Q1', 1), ('Q2', 2), ('Q3', 3), ('Q4', 4)],
    value=0,
    description='Quarter:',
    style={'description_width': 'initial'},
)

ms_output = widgets.Output()

def update_market_structure(*_):
    metric = ms_metric.value
    lcc_filter = ms_lcc_filter.value
    q = ms_quarter.value

    subset = df.copy()

    if lcc_filter == 'lcc':
        subset = subset[subset['lcc_present']]
    elif lcc_filter == 'legacy':
        subset = subset[~subset['lcc_present']]

    if q > 0:
        subset = subset[subset['quarter'] == q]

    with ms_output:
        clear_output(wait=True)

        if subset.empty:
            print("No data matches the current filters.")
            return

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Left: by market structure
        struct_means = subset.groupby('market_structure', observed=True)[metric].mean().reset_index()
        sns.barplot(data=struct_means, x='market_structure', y=metric,
                    hue='market_structure', palette='Reds', ax=axes[0], legend=False)
        y_label = 'Average Fare ($)' if metric == 'fare' else 'Fare per Mile ($)'
        axes[0].set_title(f'{y_label} by Market Structure', fontsize=14, fontweight='bold')
        axes[0].set_ylabel(y_label, fontsize=12)
        axes[0].set_xlabel('')
        for c in axes[0].containers:
            fmt = '${:.2f}' if metric == 'fare' else '${:.4f}'
            axes[0].bar_label(c, fmt=fmt, padding=3)

        # Right: LCC vs Legacy on dominant routes
        dominant = subset[subset['market_structure'] == 'Dominant (>75%)']
        if not dominant.empty:
            dom_means = dominant.groupby('dominant_type')[metric].mean().reset_index()
            sns.barplot(data=dom_means, x='dominant_type', y=metric,
                        hue='dominant_type', palette='coolwarm', ax=axes[1], legend=False)
            axes[1].set_title(f'{y_label}: LCC vs Legacy (Dominant Routes)',
                              fontsize=14, fontweight='bold')
            axes[1].set_ylabel(y_label, fontsize=12)
            axes[1].set_xlabel('')
            for c in axes[1].containers:
                fmt = '${:.2f}' if metric == 'fare' else '${:.4f}'
                axes[1].bar_label(c, fmt=fmt, padding=3)
        else:
            axes[1].text(0.5, 0.5, 'No dominant routes\nfor this filter',
                         ha='center', va='center', fontsize=14, transform=axes[1].transAxes)
            axes[1].set_title('Dominant Route Breakdown', fontweight='bold')

        plt.tight_layout()
        plt.show()

        print(f"\nShowing {len(subset):,} routes  |  "
              f"Filter: {lcc_filter}  |  "
              f"Quarter: {'All' if q == 0 else f'Q{q}'}")

        summary = subset.groupby('market_structure', observed=True)[metric].describe()[
            ['count', 'mean', 'std', 'min', 'max']
        ]
        display(summary.style.format({
            'mean': '{:.4f}', 'std': '{:.4f}', 'min': '{:.4f}', 'max': '{:.4f}'
        }))

ms_metric.observe(update_market_structure, names='value')
ms_lcc_filter.observe(update_market_structure, names='value')
ms_quarter.observe(update_market_structure, names='value')

display(widgets.VBox([ms_metric, widgets.HBox([ms_lcc_filter, ms_quarter])]))
display(ms_output)
update_market_structure()

Output()

In [5]:
# ═══════════════════════════════════════════════════════════════════════
# PANEL 4 — Fare Predictor
# Use the trained Random Forest model to predict fares for custom
# route parameters.  Adjust the sliders and see the result update.
# ═══════════════════════════════════════════════════════════════════════

pred_distance = widgets.IntSlider(
    value=1000, min=50, max=5000, step=50,
    description='Distance (mi):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='60%')
)

pred_large_ms = widgets.FloatSlider(
    value=0.50, min=0.0, max=1.0, step=0.01,
    description='Largest carrier share:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='60%'),
    readout_format='.0%'
)

pred_lf_ms = widgets.FloatSlider(
    value=0.20, min=0.0, max=1.0, step=0.01,
    description='LCC market share:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='60%'),
    readout_format='.0%'
)

pred_passengers = widgets.IntSlider(
    value=3000, min=100, max=20000, step=100,
    description='Passengers:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='60%')
)

pred_quarter = widgets.Dropdown(
    options=[('Q1', 1), ('Q2', 2), ('Q3', 3), ('Q4', 4)],
    value=2,
    description='Quarter:',
    style={'description_width': 'initial'},
)

pred_output = widgets.Output()

def update_prediction(*_):
    input_df = pd.DataFrame([{
        'nsmiles': pred_distance.value,
        'large_ms': pred_large_ms.value,
        'lf_ms': pred_lf_ms.value,
        'passengers': pred_passengers.value,
        'quarter': pred_quarter.value,
    }])

    predicted_fare = rf_model.predict(input_df)[0]

    similar = df[
        (df['nsmiles'].between(pred_distance.value * 0.8, pred_distance.value * 1.2)) &
        (df['quarter'] == pred_quarter.value)
    ]
    actual_median = similar['fare'].median() if not similar.empty else None

    with pred_output:
        clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(8, 4))

        bars = ax.barh(
            ['Model Prediction', 'Actual Median\n(similar routes)'],
            [predicted_fare, actual_median if actual_median else 0],
            color=['steelblue', 'coral'],
            height=0.4
        )

        ax.set_xlabel('Fare ($)', fontsize=12)
        ax.set_title('Predicted vs Actual Fares', fontsize=14, fontweight='bold')
        ax.set_xlim(0, max(predicted_fare, actual_median or 0) * 1.3)

        ax.bar_label(bars, fmt='$%.2f', padding=5, fontsize=11)

        if actual_median is None:
            ax.text(0.5, 0.15,
                    '(No similar routes in dataset for comparison)',
                    ha='center', transform=ax.transAxes,
                    fontsize=10, color='gray')

        plt.tight_layout()
        plt.show()

        fare_per_mile = predicted_fare / pred_distance.value if pred_distance.value > 0 else 0
        ms_label = (
            'Competitive' if pred_large_ms.value < 0.5
            else 'Moderate' if pred_large_ms.value < 0.75
            else 'Dominant'
        )
        lcc_label = 'High' if pred_lf_ms.value > 0.3 else 'Moderate' if pred_lf_ms.value > 0.1 else 'Low'

        print(f"  Predicted fare:       ${predicted_fare:.2f}")
        print(f"  Predicted $/mile:     ${fare_per_mile:.4f}")
        print(f"  Market structure:     {ms_label}")
        print(f"  LCC penetration:      {lcc_label}")
        if actual_median is not None:
            diff = predicted_fare - actual_median
            print(f"  vs. actual median:    {'+' if diff > 0 else ''}{diff:.2f} "
                  f"({'above' if diff > 0 else 'below'} median)")
            print(f"  Similar routes found: {len(similar):,}")

for w in [pred_distance, pred_large_ms, pred_lf_ms, pred_passengers, pred_quarter]:
    w.observe(update_prediction, names='value')

display(widgets.VBox([
    pred_distance, pred_large_ms, pred_lf_ms,
    pred_passengers, pred_quarter
]))
display(pred_output)
update_prediction()

Output()